# Avito: LoRA + hybrid candidate generation (submitted 0.64)

Цель — для каждого запроса вернуть до 50 `item_id` с максимальным Recall@50.
Этот notebook воспроизводит пайплайн от исходных parquet-файлов до
`/kaggle/working/answer.csv`, соответствующий лучшему отправленному варианту
с public score **0.64**.

Каскад: leakage-safe split → ML-Embed-0.6B + LoRA → exact dense top-200 →
word/char TF-IDF union → category/location-aware linear reranker → `answer.csv`.

### Среда

- Kaggle, accelerator **2×T4**, internet on;
- фиксированные seed, версии библиотек, число train pairs и шагов;
- никаких внешних inference API.


In [ ]:
!pip uninstall -y -q torchao
!pip install -q transformers==4.57.1 peft==0.17.1 accelerate==1.10.1
!pip install -q numpy==2.0.2 pandas==2.2.3 scipy==1.15.2 scikit-learn==1.6.1 pyarrow==19.0.1 tqdm==4.67.1

In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import re
import shutil
import subprocess
import time
import zipfile

import numpy as np
import pandas as pd
import requests
import torch
from tqdm.auto import tqdm

SEED = 20260927
MODEL_ID = "codefuse-ai/ML-Embed-0.6B"

TRAIN_PAIRS = 30_000
TRAIN_BATCH_PER_GPU = 16
EPOCHS = 1
LR = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
TEMPERATURE = 0.05

TRAIN_MAX_LENGTH = 128
INFER_MAX_LENGTH = 256

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

VALIDATION_GROUPS = 3200
TOP_K = 200
SEARCH_BATCH = 64

ENCODE_BATCH_CANDIDATES = [64, 96, 128]

working = Path("/kaggle/working")
input_root = Path("/kaggle/input")

assert torch.cuda.is_available(), "Enable Kaggle GPU"

print("torch:", torch.__version__)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, torch.cuda.get_device_name(i), f"{p.total_memory / 2**30:.1f} GB")

if torch.cuda.device_count() < 2:
    print("WARNING: notebook is optimized for 2 GPUs; only one GPU is visible.")

## 1. Load task data

In [ ]:
def find_input(name):
    found = list(input_root.rglob(name))
    return found[0] if found else None

train_path = find_input("train.parquet")
queries_path = find_input("benchmark_queries.parquet")
items_path = find_input("benchmark_items.parquet")

if not all([train_path, queries_path, items_path]):
    outer = working / "dataset_outer.zip"

    api = requests.get(
        "https://cloud-api.yandex.net/v1/disk/public/resources/download",
        params={"public_key": "https://disk.yandex.ru/d/sNhfo0YOjGtufg"},
        timeout=60,
    )
    api.raise_for_status()

    with requests.get(api.json()["href"], stream=True, timeout=120) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        chunk_bytes = 8 * 1024 * 1024

        with outer.open("wb") as f:
            chunks = response.iter_content(chunk_bytes)
            for chunk in tqdm(
                chunks,
                total=math.ceil(total / chunk_bytes) if total else None,
                desc="download",
                unit="chunk",
            ):
                if chunk:
                    f.write(chunk)

    root = working / "avito_data"
    root.mkdir(exist_ok=True)

    pending = [outer]
    seen = set()

    for depth in range(3):
        current = [p for p in pending if str(p) not in seen]

        for archive_path in current:
            target = root / f"layer_{depth}_{archive_path.stem}"
            target.mkdir(parents=True, exist_ok=True)

            with zipfile.ZipFile(archive_path) as archive:
                for member in tqdm(
                    archive.infolist(),
                    desc=f"unzip-{depth}",
                    unit="file",
                ):
                    archive.extract(member, target)

            seen.add(str(archive_path))

        if list(root.rglob("train.parquet")):
            break

        pending = list(root.rglob("*.zip"))

    train_path = next(root.rglob("train.parquet"))
    queries_path = next(root.rglob("benchmark_queries.parquet"))
    items_path = next(root.rglob("benchmark_items.parquet"))

print(train_path)
print(queries_path)
print(items_path)

## 2. Leakage-safe split and texts

In [ ]:
query_cols = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

item_text_cols = [
    "item_title_raw",
    "item_infm_params_text",
    "item_description_raw",
]

train = pd.read_parquet(
    train_path,
    columns=query_cols + ["item_id"] + item_text_cols,
)
items = pd.read_parquet(items_path)
queries = pd.read_parquet(queries_path)

# Critical: same dtype before membership checks.
train["item_id"] = train["item_id"].astype(str)
items["item_id"] = items["item_id"].astype(str)

item_ids = items.item_id.to_numpy()
item_to_idx = {x: i for i, x in enumerate(item_ids)}
corpus_ids = set(item_ids)

# Full query context group for both validation and train exclusion.
train["_group"] = (
    pd.util.hash_pandas_object(
        train[query_cols].fillna(""),
        index=False,
    )
    .astype(str)
)

# Validation labels exist only for clicks whose item is in benchmark corpus.
val_rows = train.loc[
    train.item_id.isin(corpus_ids),
    query_cols + ["item_id", "_group"],
].copy()

relevant = val_rows.groupby("_group").item_id.agg(set)

validation = (
    val_rows
    .drop_duplicates("_group")
    .set_index("_group")
)

validation["relevant"] = relevant

validation = validation.sample(
    min(VALIDATION_GROUPS, len(validation)),
    random_state=SEED,
)

validation["relevant_idx"] = validation.relevant.map(
    lambda xs: {item_to_idx[x] for x in xs}
)

val_groups = set(validation.index.astype(str))

# Domain-adaptation pairs:
# 1) positive item must NOT be in benchmark corpus
# 2) full query context must NOT be a validation query context
pool = train.loc[
    (~train.item_id.isin(corpus_ids))
    & (~train["_group"].isin(val_groups)),
    query_cols + item_text_cols + ["item_id", "_group"],
].copy()

# Remove exact duplicate positive pairs.
pool = pool.drop_duplicates(["_group", "item_id"])

# Conservative setup for Multiple-Negatives training:
# keep one exact query context and one exact positive item only once.
# This avoids trivial exact false negatives inside global batches.
pool = pool.sample(frac=1.0, random_state=SEED)
pool = pool.drop_duplicates("_group")
pool = pool.drop_duplicates("item_id")

pool = pool.sample(
    min(TRAIN_PAIRS, len(pool)),
    random_state=SEED,
).reset_index(drop=True)

print("train source rows:", len(train))
print("benchmark items:", len(items))
print("validation groups:", len(validation))
print("candidate clean train pairs:", len(pool))
print("train/val group overlap:", len(set(pool["_group"]) & val_groups))
print("train positive / benchmark item overlap:", pool.item_id.isin(corpus_ids).sum())

assert len(set(pool["_group"]) & val_groups) == 0
assert pool.item_id.isin(corpus_ids).sum() == 0
assert pool["_group"].is_unique
assert pool["item_id"].is_unique

In [ ]:
def clean_series(s):
    return (
        s.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

QUERY_PROMPT = (
    "Instruct: Given a Russian user search query for services on a classifieds "
    "marketplace, retrieve advertisements that satisfy the service intent and "
    "textual filters.\nQuery: "
)

def query_text(frame):
    q = clean_series(frame.search_query)
    p = clean_series(frame.search_infm_params_text)
    base = q + (" | filters: " + p).where(p.ne(""), "")
    return (QUERY_PROMPT + base).tolist()

def document_text(frame):
    title = clean_series(frame.item_title_raw)
    params = clean_series(frame.item_infm_params_text)

    # Keep the same richer text as the dense baseline.
    # TRAIN_MAX_LENGTH will truncate during training.
    desc = (
        clean_series(frame.item_description_raw)
        .str.slice(0, 1800)
    )

    return (
        "title: " + title
        + (" | parameters: " + params).where(params.ne(""), "")
        + (" | description: " + desc).where(desc.ne(""), "")
    ).tolist()

train_queries = query_text(pool)
train_docs = document_text(pool)

item_texts = document_text(items)
val_texts = query_text(validation)
benchmark_texts = query_text(queries)
all_query_texts = val_texts + benchmark_texts

train_pairs_path = working / "lora_train_pairs.parquet"

pd.DataFrame({
    "query": train_queries,
    "document": train_docs,
}).to_parquet(train_pairs_path, index=False)

item_text_path = working / "lora_item_texts.parquet"
query_text_path = working / "lora_query_texts.parquet"

pd.DataFrame({"text": item_texts}).to_parquet(item_text_path, index=False)
pd.DataFrame({"text": all_query_texts}).to_parquet(query_text_path, index=False)

print("train pairs:", len(train_queries))
print("documents:", len(item_texts))
print("queries:", len(all_query_texts))
print("saved:", train_pairs_path)

del pool, train, val_rows, relevant
gc.collect()

## 3. LoRA-дообучение на 2×T4

Берём 30 000 leakage-safe positive pairs. Одна эпоха при batch 16 на GPU
даёт ровно **937 optimizer steps**; остановки по wall-clock time нет.
Обучаются только LoRA-адаптеры attention-проекций; negatives собираются
из global batch обеих GPU.


In [ ]:
train_script = r"""
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn.functional as F

from peft import LoraConfig, TaskType, get_peft_model
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.distributed.nn.functional import all_gather as autograd_all_gather

from transformers import (
    AutoModel,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)

SEED = __SEED__
MODEL_ID = "__MODEL_ID__"
TRAIN_FILE = "__TRAIN_FILE__"
OUT_DIR = "__OUT_DIR__"

BATCH = __BATCH__
EPOCHS = __EPOCHS__
LR = __LR__
WEIGHT_DECAY = __WD__
WARMUP_RATIO = __WARMUP__
MAX_LENGTH = __MAXLEN__
TEMPERATURE = __TEMP__

LORA_R = __LORA_R__
LORA_ALPHA = __LORA_ALPHA__
LORA_DROPOUT = __LORA_DROPOUT__


class PairDataset(Dataset):
    def __init__(self, path):
        df = pd.read_parquet(path)
        self.q = df["query"].tolist()
        self.d = df["document"].tolist()

    def __len__(self):
        return len(self.q)

    def __getitem__(self, i):
        return self.q[i], self.d[i]


def pool_last(outputs, attention_mask):
    positions = attention_mask.sum(dim=1) - 1
    rows = torch.arange(len(positions), device=positions.device)
    return outputs.last_hidden_state[rows, positions]


def main():
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = int(os.environ["RANK"])
    world = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    dist.init_process_group("nccl")

    seed = SEED + rank
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    major, _ = torch.cuda.get_device_capability(local_rank)
    dtype = torch.bfloat16 if major >= 8 else torch.float16

    if rank == 0:
        print("=" * 72, flush=True)
        print("DDP TRAIN", flush=True)
        print("world_size:", world, flush=True)
        print("batch_per_gpu:", BATCH, flush=True)
        print("global contrastive batch:", BATCH * world, flush=True)
        print("dtype:", dtype, flush=True)
        print("=" * 72, flush=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    base = AutoModel.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
    ).to(device)

    base.config.use_cache = False

    try:
        base.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    except TypeError:
        base.gradient_checkpointing_enable()

    base.enable_input_require_grads()

    lora = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    model = get_peft_model(base, lora)

    if rank == 0:
        model.print_trainable_parameters()

    model = DDP(
        model,
        device_ids=[local_rank],
        output_device=local_rank,
        find_unused_parameters=False,
    )

    dataset = PairDataset(TRAIN_FILE)

    sampler = DistributedSampler(
        dataset,
        num_replicas=world,
        rank=rank,
        shuffle=True,
        seed=SEED,
        drop_last=True,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH,
        sampler=sampler,
        num_workers=0,
        pin_memory=True,
        drop_last=True,
    )

    trainable = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = len(loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    # T4 -> FP16, so use GradScaler.
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(dtype == torch.float16),
    )

    started = time.time()
    running = []

    model.train()

    for epoch in range(EPOCHS):
        sampler.set_epoch(epoch)

        for step, batch in enumerate(loader, 1):
            qs, ds = batch

            q = tokenizer(
                list(qs),
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            ).to(device, non_blocking=True)

            d = tokenizer(
                list(ds),
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            ).to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                "cuda",
                dtype=dtype,
                enabled=True,
            ):
                qe = F.normalize(
                    pool_last(model(**q), q.attention_mask),
                    dim=1,
                )

                de = F.normalize(
                    pool_last(model(**d), d.attention_mask),
                    dim=1,
                )

                # Autograd-aware gather: negatives from BOTH GPUs.
                all_q = torch.cat(
                    autograd_all_gather(qe),
                    dim=0,
                )

                all_d = torch.cat(
                    autograd_all_gather(de),
                    dim=0,
                )

                labels = (
                    rank * BATCH
                    + torch.arange(BATCH, device=device)
                )

                q_logits = qe @ all_d.T / TEMPERATURE
                d_logits = de @ all_q.T / TEMPERATURE

                loss = (
                    F.cross_entropy(q_logits, labels)
                    + F.cross_entropy(d_logits, labels)
                ) / 2

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            # Mean loss for logging.
            loss_log = loss.detach().clone()
            dist.all_reduce(loss_log, op=dist.ReduceOp.SUM)
            loss_log /= world
            running.append(float(loss_log))

            if rank == 0 and (
                step == 1
                or step % 25 == 0
                or step == len(loader)
            ):
                elapsed = time.time() - started
                done = epoch * len(loader) + step
                total = len(loader) * EPOCHS
                eta = elapsed / done * (total - done)

                print(
                    f"TRAIN {done}/{total} "
                    f"loss={np.mean(running[-25:]):.4f} "
                    f"lr={scheduler.get_last_lr()[0]:.2e} "
                    f"elapsed={elapsed/60:.1f}m "
                    f"eta={eta/60:.1f}m",
                    flush=True,
                )

    dist.barrier()

    if rank == 0:
        out = Path(OUT_DIR)
        out.mkdir(parents=True, exist_ok=True)

        model.module.save_pretrained(out)
        tokenizer.save_pretrained(out)

        report = {
            "model": MODEL_ID,
            "method": "LoRA DDP symmetric global in-batch contrastive",
            "world_size": world,
            "batch_per_gpu": BATCH,
            "global_contrastive_batch": BATCH * world,
            "epochs": EPOCHS,
            "lr": LR,
            "warmup_ratio": WARMUP_RATIO,
            "train_max_length": MAX_LENGTH,
            "temperature": TEMPERATURE,
            "training_minutes": (time.time() - started) / 60,
            "final_loss_25": float(np.mean(running[-25:])),
        }

        Path("/kaggle/working/lora_train_report.json").write_text(
            json.dumps(report, indent=2),
            encoding="utf-8",
        )

        print(json.dumps(report, indent=2), flush=True)
        print("adapter saved:", out, flush=True)

    dist.destroy_process_group()


if __name__ == "__main__":
    main()
"""

repls = {
    "__SEED__": str(SEED),
    "__MODEL_ID__": MODEL_ID,
    "__TRAIN_FILE__": str(train_pairs_path),
    "__OUT_DIR__": str(working / "ml_embed_lora_adapter"),
    "__BATCH__": str(TRAIN_BATCH_PER_GPU),
    "__EPOCHS__": str(EPOCHS),
    "__LR__": repr(LR),
    "__WD__": repr(WEIGHT_DECAY),
    "__WARMUP__": repr(WARMUP_RATIO),
    "__MAXLEN__": str(TRAIN_MAX_LENGTH),
    "__TEMP__": repr(TEMPERATURE),
    "__LORA_R__": str(LORA_R),
    "__LORA_ALPHA__": str(LORA_ALPHA),
    "__LORA_DROPOUT__": repr(LORA_DROPOUT),
}

for k, v in repls.items():
    train_script = train_script.replace(k, v)

train_script_path = working / "train_lora_ddp.py"
train_script_path.write_text(train_script, encoding="utf-8")

print(train_script_path)
print(train_script[:1200])

In [ ]:
train_started = time.time()

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    str(working / "train_lora_ddp.py"),
]

print("RUN:", " ".join(cmd), flush=True)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in proc.stdout:
    print(line, end="", flush=True)

ret = proc.wait()

if ret != 0:
    raise RuntimeError(f"torchrun failed with exit code {ret}")

print(
    "Training wall time:",
    f"{(time.time() - train_started)/60:.1f} min",
)

## 4. 2×T4 re-encoding with the trained adapter

In [ ]:
encode_script = r"""
import gc
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn.functional as F

from peft import PeftModel
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "__MODEL_ID__"
ADAPTER_DIR = "__ADAPTER__"

ITEM_FILE = "__ITEM_FILE__"
QUERY_FILE = "__QUERY_FILE__"

MAX_LENGTH = __MAXLEN__
BATCH_CANDIDATES = __BATCHES__


def pool_last(outputs, attention_mask):
    pos = attention_mask.sum(dim=1) - 1
    rows = torch.arange(len(pos), device=pos.device)
    return outputs.last_hidden_state[rows, pos]


@torch.inference_mode()
def bench(model, tokenizer, texts, device, dtype, bs):
    n = min(1024, len(texts))
    t0 = time.time()

    for start in range(0, n, bs):
        batch = tokenizer(
            texts[start:start+bs],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        with torch.autocast("cuda", dtype=dtype):
            emb = F.normalize(
                pool_last(model(**batch), batch.attention_mask),
                dim=1,
            )

    torch.cuda.synchronize(device)
    sec = time.time() - t0
    return n / sec


@torch.inference_mode()
def encode(model, tokenizer, texts, device, dtype, bs, name, rank):
    parts = []
    total_batches = math.ceil(len(texts) / bs)
    stage = time.time()

    for step, start in enumerate(range(0, len(texts), bs), 1):
        batch = tokenizer(
            texts[start:start+bs],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=dtype):
            emb = F.normalize(
                pool_last(model(**batch), batch.attention_mask),
                dim=1,
            )

        parts.append(
            emb.cpu().to(torch.float16).numpy()
        )

        if step == 1 or step % 25 == 0 or step == total_batches:
            elapsed = time.time() - stage
            eta = elapsed / step * (total_batches - step)
            print(
                f"RANK{rank} {name} {step}/{total_batches} "
                f"elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m",
                flush=True,
            )

    return np.vstack(parts)


def shard_bounds(n, rank, world):
    start = n * rank // world
    end = n * (rank + 1) // world
    return start, end


def main():
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
    dist.init_process_group("nccl")

    major, _ = torch.cuda.get_device_capability(local_rank)
    dtype = torch.bfloat16 if major >= 8 else torch.float16

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    base = AutoModel.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
    ).to(device)

    model = PeftModel.from_pretrained(
        base,
        ADAPTER_DIR,
    )

    model = model.merge_and_unload()
    model.eval()
    model.config.use_cache = True

    item_all = pd.read_parquet(ITEM_FILE)["text"].tolist()
    query_all = pd.read_parquet(QUERY_FILE)["text"].tolist()

    is_, ie = shard_bounds(len(item_all), rank, world)
    qs, qe = shard_bounds(len(query_all), rank, world)

    item_texts = item_all[is_:ie]
    query_texts = query_all[qs:qe]

    # Autotune encoding batch independently on each identical T4.
    results = []

    for bs in BATCH_CANDIDATES:
        try:
            torch.cuda.empty_cache()
            speed = bench(
                model, tokenizer, item_texts, device, dtype, bs
            )

            print(
                f"RANK{rank} BENCH batch={bs} speed={speed:.2f} docs/s",
                flush=True,
            )

            results.append((speed, bs))

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(
                    f"RANK{rank} BENCH batch={bs} OOM",
                    flush=True,
                )
                torch.cuda.empty_cache()
            else:
                raise

    if not results:
        raise RuntimeError("No encoding batch size fits")

    best_speed, best_bs = max(results)

    print(
        f"RANK{rank} SELECTED batch={best_bs} "
        f"speed={best_speed:.2f}",
        flush=True,
    )

    item_emb = encode(
        model, tokenizer, item_texts,
        device, dtype, best_bs, "ITEMS", rank
    )

    query_emb = encode(
        model, tokenizer, query_texts,
        device, dtype, best_bs, "QUERIES", rank
    )

    np.save(
        f"/kaggle/working/item_emb_rank{rank}.npy",
        item_emb,
    )

    np.save(
        f"/kaggle/working/query_emb_rank{rank}.npy",
        query_emb,
    )

    Path(
        f"/kaggle/working/encode_meta_rank{rank}.json"
    ).write_text(
        json.dumps({
            "rank": rank,
            "item_start": is_,
            "item_end": ie,
            "query_start": qs,
            "query_end": qe,
            "batch": best_bs,
            "bench_speed": best_speed,
        }, indent=2),
        encoding="utf-8",
    )

    dist.barrier()

    if rank == 0:
        print("ENCODING DONE ON BOTH GPUS", flush=True)

    dist.destroy_process_group()


if __name__ == "__main__":
    main()
"""

encode_script = (
    encode_script
    .replace("__MODEL_ID__", MODEL_ID)
    .replace("__ADAPTER__", str(working / "ml_embed_lora_adapter"))
    .replace("__ITEM_FILE__", str(item_text_path))
    .replace("__QUERY_FILE__", str(query_text_path))
    .replace("__MAXLEN__", str(INFER_MAX_LENGTH))
    .replace("__BATCHES__", repr(ENCODE_BATCH_CANDIDATES))
)

encode_script_path = working / "encode_lora_ddp.py"
encode_script_path.write_text(encode_script, encoding="utf-8")

print(encode_script_path)

In [ ]:
encode_started = time.time()

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    str(working / "encode_lora_ddp.py"),
]

print("RUN:", " ".join(cmd), flush=True)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in proc.stdout:
    print(line, end="", flush=True)

ret = proc.wait()

if ret != 0:
    raise RuntimeError(f"encoding torchrun failed with exit code {ret}")

print(
    "Encoding wall time:",
    f"{(time.time() - encode_started)/60:.1f} min",
)

In [ ]:
# Shards are contiguous, so rank0 + rank1 restores original order.
item_embeddings = np.vstack([
    np.load(working / f"item_emb_rank{r}.npy")
    for r in range(2)
])

query_embeddings = np.vstack([
    np.load(working / f"query_emb_rank{r}.npy")
    for r in range(2)
])

assert item_embeddings.shape[0] == len(items)
assert query_embeddings.shape[0] == len(validation) + len(queries)

print("items:", item_embeddings.shape, item_embeddings.dtype)
print("queries:", query_embeddings.shape, query_embeddings.dtype)

## 5. Exact cosine top-200

In [ ]:
item_tensor = torch.from_numpy(item_embeddings).to("cuda:0")

indices_parts = []
score_parts = []

search_total = math.ceil(
    len(query_embeddings) / SEARCH_BATCH
)

search_started = time.time()

with torch.inference_mode():
    for step, start in enumerate(
        range(0, len(query_embeddings), SEARCH_BATCH),
        1,
    ):
        q = torch.from_numpy(
            query_embeddings[start:start + SEARCH_BATCH]
        ).to("cuda:0")

        scores = q @ item_tensor.T

        values, indices = torch.topk(
            scores,
            k=TOP_K,
            dim=1,
        )

        indices_parts.append(
            indices.cpu().numpy().astype(np.int32)
        )

        score_parts.append(
            values.cpu().numpy().astype(np.float16)
        )

        if (
            step == 1
            or step % 10 == 0
            or step == search_total
        ):
            elapsed = time.time() - search_started
            eta = elapsed / step * (search_total - step)
            print(
                f"SEARCH {step}/{search_total} "
                f"elapsed={elapsed/60:.1f}m "
                f"eta={eta/60:.1f}m",
                flush=True,
            )

top_indices = np.vstack(indices_parts)
top_scores = np.vstack(score_parts)

nv = len(validation)

val_indices = top_indices[:nv]
val_scores = top_scores[:nv]

bench_indices = top_indices[nv:]
bench_scores = top_scores[nv:]

print(top_indices.shape, top_scores.shape)

## 6. Recall and artifacts

In [ ]:
def recall_at(k):
    return float(np.mean([
        len(set(map(int, row[:k])) & rel) / len(rel)
        for row, rel in zip(
            val_indices,
            validation.relevant_idx,
        )
    ]))

metrics = {
    f"recall@{k}": recall_at(k)
    for k in (50, 100, 200)
}

print(json.dumps(metrics, indent=2))

In [ ]:
output_npz = working / "ml_embed_lora_2gpu_top200.npz"

np.savez_compressed(
    output_npz,
    model_id=np.asarray([MODEL_ID]),
    val_group_ids=validation.index.astype(str).to_numpy(),
    val_indices=val_indices,
    val_scores=val_scores,
    benchmark_query_ids=queries.query_id.astype(str).to_numpy(),
    benchmark_indices=bench_indices,
    benchmark_scores=bench_scores,
    item_ids=item_ids,
)

dense_answer = pd.DataFrame({
    "query_id": queries.query_id.astype(str),
    "answer": [
        " ".join(item_ids[row[:50]])
        for row in bench_indices
    ],
})

dense_answer.to_csv(
    working / "lora_dense_answer.csv",
    index=False,
)

train_report_path = working / "lora_train_report.json"

train_report = (
    json.loads(train_report_path.read_text())
    if train_report_path.exists()
    else {}
)

report = {
    "model": MODEL_ID,
    "method": "LoRA 2xT4 DDP + global in-batch negatives",
    "train_pairs": len(train_queries),
    "train_batch_per_gpu": TRAIN_BATCH_PER_GPU,
    "global_contrastive_batch": TRAIN_BATCH_PER_GPU * 2,
    "epochs": EPOCHS,
    "lr": LR,
    "train_max_length": TRAIN_MAX_LENGTH,
    "infer_max_length": INFER_MAX_LENGTH,
    "temperature": TEMPERATURE,
    "metrics": metrics,
    "training": train_report,
}

report_path = working / "lora_report.json"

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

assert dense_answer.shape == (len(queries), 2)
assert dense_answer.query_id.is_unique
assert dense_answer.answer.str.split().str.len().eq(50).all()

print(json.dumps(report, indent=2))
print()
print("Wrote:", output_npz)
print("Wrote:", working / "lora_dense_answer.csv")
print("Wrote:", report_path)
print("Adapter:", working / "ml_embed_lora_adapter")

## Что скачивать из Kaggle Output

Главные результаты:

1. `ml_embed_lora_2gpu_top200.npz` — для fusion;
2. `lora_report.json` — метрики;
3. `lora_train_report.json` — обучение;
4. `ml_embed_lora_adapter/` — обученный LoRA adapter;
5. `lora_dense_answer.csv` — dense-only диагностика.

После проверки метрик временные `.npy` shard-файлы можно удалить.

## 7. Анализ ошибок и hybrid retrieval

| Тип ошибки | Как исправлено |
|---|---|
| Семантическое совпадение без общих слов | multilingual dense retrieval |
| Опечатки, словоформы, слитное написание | char TF-IDF 3–5-grams |
| Точные названия и числа | word TF-IDF 1–2-grams |
| Семантически похожее объявление из другого города | exact-location, transition feature, local quota |
| Игнорирование фильтров | params в query text + отдельный lexical channel |
| Один retrieval-канал не покрывает все случаи | candidate union + linear reranker |

Ниже запускается тот же leakage-safe hybrid/reranker, который использован
для отправки 0.64.


In [ ]:
"""Leakage-safe experiments with hybrid candidate retrieval.

This script intentionally does not write submission files.  It uses clicked
train rows whose items are present in benchmark_items as pseudo-test labels.
All behavioural profiles are learned only from train rows whose items are NOT
in benchmark_items, so the relevant item itself cannot leak into a profile.
"""

from __future__ import annotations

from collections import defaultdict
import gc
import json
from pathlib import Path
import re
import time
import unicodedata

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier


DATA = train_path.parent
OUT = working / "hybrid_lora_results.json"
DENSE = output_npz
SEED = 20260927
N_GROUPS = 3200
RETRIEVAL_K = 180
TOP_K = 50


def normalize(value) -> str:
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value)).lower().replace("ё", "е")
    value = re.sub(r"[^0-9a-zа-я]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def clean(series: pd.Series) -> pd.Series:
    return series.fillna("").map(normalize)


def sparse_topk(qmat, imat, k=RETRIEVAL_K, batch=64):
    """Return per-query item indices and cosine scores."""
    all_idx, all_val = [], []
    transposed = imat.T.tocsc()
    for start in range(0, qmat.shape[0], batch):
        scores = (qmat[start : start + batch] @ transposed).tocsr()
        for row in range(scores.shape[0]):
            left, right = scores.indptr[row], scores.indptr[row + 1]
            idx = scores.indices[left:right]
            val = scores.data[left:right]
            if len(val) > k:
                keep = np.argpartition(val, -k)[-k:]
                keep = keep[np.argsort(val[keep])[::-1]]
                idx, val = idx[keep], val[keep]
            elif len(val):
                order = np.argsort(val)[::-1]
                idx, val = idx[order], val[order]
            all_idx.append(idx.astype(np.int32, copy=False))
            all_val.append(val.astype(np.float32, copy=False))
    return all_idx, all_val


def rrf_score(channels, weights, rrf_k=60):
    result = defaultdict(float)
    for ranking, weight in zip(channels, weights):
        for rank, idx in enumerate(ranking):
            result[int(idx)] += weight / (rrf_k + rank + 1)
    return result


def rank_from_scores(scores, limit=TOP_K):
    return [idx for idx, _ in sorted(scores.items(), key=lambda pair: pair[1], reverse=True)[:limit]]


def recall(ranking, relevant):
    return len(set(ranking[:TOP_K]) & relevant) / len(relevant)


def add_structured(
    scores,
    row,
    normalized_query,
    item_locs,
    item_microcats,
    transition_probs,
    exact_micro_probs,
    centroid_scores,
    popularity,
    exact_loc_bonus=0.0,
    transition_bonus=0.0,
    exact_micro_bonus=0.0,
    centroid_micro_bonus=0.0,
    popularity_bonus=0.0,
):
    source_loc = int(row["search_location_id"])
    transitions = transition_probs.get(source_loc, {})
    exact_micro = exact_micro_probs.get(normalized_query, {})
    centroid_micro = centroid_scores
    for idx in scores:
        loc = int(item_locs[idx])
        microcat = int(item_microcats[idx])
        if loc == source_loc:
            scores[idx] += exact_loc_bonus
        if transition_bonus:
            scores[idx] += transition_bonus * transitions.get(loc, 0.0)
        if exact_micro_bonus:
            scores[idx] += exact_micro_bonus * exact_micro.get(microcat, 0.0)
        if centroid_micro_bonus:
            scores[idx] += centroid_micro_bonus * centroid_micro.get(microcat, 0.0)
        if popularity_bonus:
            scores[idx] += popularity_bonus * popularity[idx]
    return scores


def quota_ranking(base_ranking, item_locs, query_loc, local_quota):
    local = [idx for idx in base_ranking if item_locs[idx] == query_loc]
    chosen = local[:local_quota]
    used = set(chosen)
    for idx in base_ranking:
        if idx not in used:
            chosen.append(idx)
            used.add(idx)
        if len(chosen) == TOP_K:
            break
    return chosen


started = time.time()
print("Loading parquet files", flush=True)
train_columns = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category", "item_id",
    "item_microcat_id", "item_location_id",
]
train = pd.read_parquet(DATA / "train.parquet", columns=train_columns)
items = pd.read_parquet(DATA / "benchmark_items.parquet")
item_ids = items["item_id"].astype(str).to_numpy()
item_id_to_idx = {value: idx for idx, value in enumerate(item_ids)}
benchmark_item_ids = set(item_ids)
item_locs = items["item_location_id"].to_numpy()
item_microcats = items["item_microcat_id"].to_numpy()

# Validation labels: exact query feature tuples, with all clicked benchmark
# items for the tuple treated as relevant.
query_cols = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
val_rows = train.loc[train["item_id"].isin(benchmark_item_ids), query_cols + ["item_id"]].copy()
val_rows["_group"] = pd.util.hash_pandas_object(val_rows[query_cols].fillna(""), index=False).astype(str)
relevant_ids = val_rows.groupby("_group")["item_id"].agg(set)
validation = val_rows.drop_duplicates("_group").set_index("_group")
validation["relevant"] = relevant_ids
validation = validation.sample(min(N_GROUPS, len(validation)), random_state=SEED)
validation["relevant_idx"] = validation["relevant"].map(
    lambda values: {item_id_to_idx[value] for value in values}
)
split = len(validation) // 2
tune_positions = np.arange(split)
test_positions = np.arange(split, len(validation))

val_main = clean(validation["search_query"])
val_params = clean(validation["search_infm_params_text"])
val_mixed_text = val_main + " " + val_main + " " + val_params

# Leakage-safe historical rows.  These can transfer intent and geography but
# contain none of the validation corpus items.
profile = train.loc[
    ~train["item_id"].isin(benchmark_item_ids),
    ["search_query", "search_location_id", "item_location_id", "item_microcat_id"],
].copy()
profile["normalized_query"] = clean(profile["search_query"])

loc_counts = (
    profile.groupby(["search_location_id", "item_location_id"])
    .size()
    .rename("n")
    .reset_index()
)
loc_counts["total"] = loc_counts.groupby("search_location_id")["n"].transform("sum")
loc_counts["prob"] = loc_counts["n"] / loc_counts["total"]
loc_counts = loc_counts.sort_values(["search_location_id", "n"], ascending=[True, False])
# Keep top five transitions: they cover 96%+ of held-out clicked locations.
transition_probs = {
    int(source): {int(target): float(prob) for target, prob in zip(group.item_location_id, group.prob)}
    for source, group in loc_counts.groupby("search_location_id").head(5).groupby("search_location_id")
}

micro_counts = (
    profile.groupby(["normalized_query", "item_microcat_id"])
    .size()
    .rename("n")
    .reset_index()
)
micro_counts["total"] = micro_counts.groupby("normalized_query")["n"].transform("sum")
micro_counts["prob"] = micro_counts["n"] / micro_counts["total"]
micro_counts = micro_counts.sort_values(["normalized_query", "n"], ascending=[True, False])
exact_micro_probs = {
    query: {int(cat): float(prob) for cat, prob in zip(group.item_microcat_id, group.prob)}
    for query, group in micro_counts.groupby("normalized_query").head(5).groupby("normalized_query")
}

# A query-to-microcategory channel for queries without an exact history match.
# Each microcategory is represented by its most frequent historical queries.
query_cat_counts = (
    profile.groupby(["item_microcat_id", "normalized_query"])
    .size()
    .rename("n")
    .reset_index()
    .sort_values(["item_microcat_id", "n"], ascending=[True, False])
)
micro_docs = (
    query_cat_counts.groupby("item_microcat_id")
    .head(250)
    .groupby("item_microcat_id")["normalized_query"]
    .agg(" ".join)
)
micro_vec = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=80_000,
    sublinear_tf=True, dtype=np.float32,
)
micro_mat = micro_vec.fit_transform(micro_docs)
micro_qmat = micro_vec.transform(val_main)
micro_sim = (micro_qmat @ micro_mat.T).tocsr()
centroid_by_query = []
microcat_values = micro_docs.index.to_numpy()
for row in range(micro_sim.shape[0]):
    left, right = micro_sim.indptr[row], micro_sim.indptr[row + 1]
    vals, cols = micro_sim.data[left:right], micro_sim.indices[left:right]
    if len(vals) > 5:
        keep = np.argpartition(vals, -5)[-5:]
        vals, cols = vals[keep], cols[keep]
    centroid_by_query.append({int(microcat_values[col]): float(value) for col, value in zip(cols, vals)})
del micro_qmat, micro_mat, micro_vec, micro_sim

print("Preparing item texts", flush=True)
item_title = clean(items["item_title_raw"])
item_params = clean(items["item_infm_params_text"])
item_desc = clean(items["item_description_raw"]).str.slice(0, 1000)
item_mixed = item_title + " " + item_title + " " + item_params + " " + item_desc
# Raw text is no longer needed and would otherwise remain alongside every
# sparse matrix below.
items.drop(columns=["item_title_raw", "item_infm_params_text", "item_description_raw"], inplace=True)

# Fit a shared word vocabulary, but materialise only one field matrix at a
# time.  Shared IDF keeps the channels comparable without a multi-GB peak.
print("Fitting shared word index", flush=True)
word_vec = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.995,
    max_features=200_000, sublinear_tf=True, dtype=np.float32,
)
word_vec.fit(item_mixed)

channels = {}
channel_scores = {}
for name, query_text, item_text in [
    ("mixed", val_mixed_text, item_mixed),
    ("title", val_main, item_title),
    ("params", val_main + " " + val_params + " " + val_params, item_params),
    ("description", val_main, item_desc),
]:
    print(f"Retrieving {name}", flush=True)
    qm = word_vec.transform(query_text)
    im = word_vec.transform(item_text)
    channels[name], channel_scores[name] = sparse_topk(qm, im)
    del qm, im
    gc.collect()
    if name == "mixed":
        del item_mixed
        gc.collect()

del word_vec
gc.collect()

print("Fitting character index", flush=True)
char_vec = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.995,
    max_features=180_000, sublinear_tf=True, dtype=np.float32,
)
char_item = char_vec.fit_transform(item_title + " " + item_params.str.slice(0, 300))
char_query = char_vec.transform(val_main)
channels["char"], channel_scores["char"] = sparse_topk(char_query, char_item)
del char_item, char_query, char_vec
gc.collect()

# Neural retrieval was computed on Kaggle GPU over exactly the same corpus and
# validation sample.  Fail loudly on any ordering mismatch: indices in the NPZ
# are row positions of benchmark_items.parquet, not item IDs.
print("Loading ML-Embed dense candidates", flush=True)
assert DENSE.exists(), f"Missing {DENSE}; run kaggle_ml_embed_2026.ipynb first"
dense = np.load(DENSE, allow_pickle=True)  # trusted artifact produced by our Kaggle notebook
assert np.array_equal(dense["item_ids"].astype(str), item_ids)
assert np.array_equal(dense["val_group_ids"].astype(str), validation.index.astype(str).to_numpy())
assert dense["val_indices"].shape[0] == len(validation)
channels["dense"] = [row.astype(np.int32, copy=False) for row in dense["val_indices"]]
channel_scores["dense"] = [row.astype(np.float32) for row in dense["val_scores"]]

reviews = items["item_rating_reviews_count"].fillna(0).to_numpy(dtype=float)
ratings = items["item_rating"].fillna(0).to_numpy(dtype=float)
popularity = np.log1p(reviews) * np.clip(ratings / 5, 0, 1)
popularity /= max(popularity.max(), 1)
review_norm = np.log1p(reviews)
review_norm /= max(review_norm.max(), 1)

records = []


def evaluate(name, builder, positions, phase):
    values = []
    hits = 0
    totals = 0
    for pos in positions:
        ranking = builder(pos, validation.iloc[pos])
        relevant = validation.iloc[pos]["relevant_idx"]
        values.append(recall(ranking, relevant))
        hits += len(set(ranking[:TOP_K]) & relevant)
        totals += len(relevant)
    macro = float(np.mean(values))
    micro = hits / totals
    records.append({"name": name, "phase": phase, "macro_recall50": macro, "micro_recall50": micro})
    print(f"{phase:5s} {name:60s} macro={macro:.6f} micro={micro:.6f}", flush=True)
    return macro


# Individual fields and candidate-union ceiling.
for name in channels:
    evaluate(name, lambda p, r, n=name: channels[n][p], test_positions, "test")

lexical_names = ["mixed", "title", "params", "description", "char"]
all_names = lexical_names + ["dense"]
# Oracle diagnostic over the complete union (up to 5 * 180 candidates), not an
# arbitrary first 50 from an unordered set.
union_values = []
union_hits = union_total = 0
for pos in test_positions:
    union = set().union(*(set(channels[n][pos]) for n in all_names))
    relevant = validation.iloc[pos]["relevant_idx"]
    hits = len(union & relevant)
    union_values.append(hits / len(relevant))
    union_hits += hits
    union_total += len(relevant)
records.append({
    "name": "union@all-channels oracle",
    "phase": "test",
    "macro_recall50": float(np.mean(union_values)),
    "micro_recall50": union_hits / union_total,
})
print(
    f"test  {'union@all-channels oracle':60s} "
    f"macro={np.mean(union_values):.6f} micro={union_hits / union_total:.6f}",
    flush=True,
)

# Tune a compact set of fielded RRF weights on tune; report chosen variants on
# untouched test groups.  We avoid an exhaustive weight search.
weight_configs = {
    "mixed+char": [1.0, 0.8],
    "title+params+desc+char": [1.2, 0.55, 0.55, 0.8],
    "fielded-title-heavy": [1.5, 0.45, 0.35, 0.9],
    "all-five": [0.65, 1.0, 0.45, 0.35, 0.75],
    "dense+mixed+char": [1.0, 0.8, 0.8],
    "dense+all-lexical": [1.0, 0.55, 0.8, 0.35, 0.25, 0.65],
}
config_channels = {
    "mixed+char": ["mixed", "char"],
    "title+params+desc+char": ["title", "params", "description", "char"],
    "fielded-title-heavy": ["title", "params", "description", "char"],
    "all-five": lexical_names,
    "dense+mixed+char": ["dense", "mixed", "char"],
    "dense+all-lexical": ["dense", "mixed", "title", "params", "description", "char"],
}


def make_builder(config, structured=None):
    names = config_channels[config]
    weights = weight_configs[config]
    structured = structured or {}

    def builder(pos, row):
        scores = rrf_score([channels[name][pos] for name in names], weights)
        add_structured(
            scores,
            row,
            val_main.iloc[pos],
            item_locs,
            item_microcats,
            transition_probs,
            exact_micro_probs,
            centroid_by_query[pos],
            popularity,
            **structured,
        )
        return rank_from_scores(scores)

    return builder


for config in weight_configs:
    evaluate(config, make_builder(config), tune_positions, "tune")
    evaluate(config, make_builder(config), test_positions, "test")

best_config = max(
    (record for record in records if record["phase"] == "tune" and record["name"] in weight_configs),
    key=lambda record: record["macro_recall50"],
)["name"]
print("Best text config:", best_config, flush=True)

# Structured reranking ablations. RRF values are roughly 0.005--0.03, hence
# bonuses in the 1e-3 range are meaningful but remain soft.
structured_configs = {
    "exact-location": {"exact_loc_bonus": 0.004},
    "transition-location": {"transition_bonus": 0.006},
    "transition+exact": {"exact_loc_bonus": 0.0015, "transition_bonus": 0.005},
    "exact-microcat": {"exact_micro_bonus": 0.005},
    "centroid-microcat": {"centroid_micro_bonus": 0.003},
    "microcats-both": {"exact_micro_bonus": 0.005, "centroid_micro_bonus": 0.002},
    "location+microcats": {
        "exact_loc_bonus": 0.0015,
        "transition_bonus": 0.005,
        "exact_micro_bonus": 0.005,
        "centroid_micro_bonus": 0.002,
    },
    "location+microcats+pop": {
        "exact_loc_bonus": 0.0015,
        "transition_bonus": 0.005,
        "exact_micro_bonus": 0.005,
        "centroid_micro_bonus": 0.002,
        "popularity_bonus": 0.0005,
    },
}
for suffix, kwargs in structured_configs.items():
    name = f"{best_config} + {suffix}"
    evaluate(name, make_builder(best_config, kwargs), tune_positions, "tune")

best_structured_name = max(
    (
        record
        for record in records
        if record["phase"] == "tune" and record["name"].startswith(best_config + " +")
    ),
    key=lambda record: record["macro_recall50"],
)["name"]
best_suffix = best_structured_name.split(" + ", 1)[1]
best_kwargs = structured_configs[best_suffix]
evaluate(best_structured_name, make_builder(best_config, best_kwargs), test_positions, "test")

# Quota ablation on top of a larger RRF pool (top 180 before quota).
def quota_builder(quota):
    base = make_builder(best_config, best_kwargs)

    def builder(pos, row):
        # Recreate the same scores but retain a deeper list.
        names = config_channels[best_config]
        scores = rrf_score([channels[name][pos] for name in names], weight_configs[best_config])
        add_structured(
            scores, row, val_main.iloc[pos], item_locs, item_microcats,
            transition_probs, exact_micro_probs, centroid_by_query[pos], popularity,
            **best_kwargs,
        )
        deep = rank_from_scores(scores, limit=RETRIEVAL_K)
        return quota_ranking(deep, item_locs, int(row["search_location_id"]), quota)

    return builder


for quota in (20, 30, 40):
    evaluate(f"best + exact-location quota {quota}", quota_builder(quota), test_positions, "test")

# Checkpoint the expensive retrieval results before the optional supervised
# tail.  A later interruption must not erase several minutes of experiments.
OUT.write_text(
    json.dumps(
        {
            "status": "structured_checkpoint",
            "settings": {
                "seed": SEED,
                "groups": len(validation),
                "tune_groups": len(tune_positions),
                "test_groups": len(test_positions),
                "retrieval_k": RETRIEVAL_K,
                "top_k": TOP_K,
                "best_text_config": best_config,
                "best_structured_config": best_structured_name,
            },
            "results": records,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# Lightweight pointwise reranker. It learns only on tune groups and is assessed
# on untouched test groups. Candidate features are ranks/scores from each text
# channel plus the leakage-safe structural signals.
def candidate_features(pos, row):
    candidate_set = set()
    rank_maps, score_maps = {}, {}
    for name in all_names:
        ranking = channels[name][pos]
        values = channel_scores[name][pos]
        candidate_set.update(map(int, ranking))
        rank_maps[name] = {int(idx): rank for rank, idx in enumerate(ranking)}
        score_maps[name] = {int(idx): float(value) for idx, value in zip(ranking, values)}

    source_loc = int(row["search_location_id"])
    trans = transition_probs.get(source_loc, {})
    exact_micro = exact_micro_probs.get(val_main.iloc[pos], {})
    centroid_micro = centroid_by_query[pos]
    candidates = np.fromiter(candidate_set, dtype=np.int32)
    features = []
    for idx in candidates:
        values = []
        for name in all_names:
            rank = rank_maps[name].get(int(idx), RETRIEVAL_K + 50)
            values.extend([1.0 / (rank + 1), score_maps[name].get(int(idx), 0.0)])
        values.extend(
            [
                float(item_locs[idx] == source_loc),
                trans.get(int(item_locs[idx]), 0.0),
                exact_micro.get(int(item_microcats[idx]), 0.0),
                centroid_micro.get(int(item_microcats[idx]), 0.0),
                popularity[idx],
                review_norm[idx],
                ratings[idx] / 5,
            ]
        )
        features.append(values)
    return candidates, np.asarray(features, dtype=np.float32)


print("Training streaming supervised reranker", flush=True)
reranker = SGDClassifier(
    loss="log_loss", penalty="l2", alpha=3e-5, random_state=SEED,
    average=True,
)
first_batch = True
for epoch in range(2):
    for start in range(0, len(tune_positions), 64):
        batch_x, batch_y = [], []
        for pos in tune_positions[start : start + 64]:
            candidates, feats = candidate_features(pos, validation.iloc[pos])
            labels = np.fromiter(
                (idx in validation.iloc[pos]["relevant_idx"] for idx in candidates),
                dtype=np.int8,
            )
            batch_x.append(feats)
            batch_y.append(labels)
        batch_x = np.vstack(batch_x)
        batch_y = np.concatenate(batch_y)
        # Roughly one positive per several hundred hard candidates.
        sample_weight = np.where(batch_y == 1, 120.0, 1.0)
        kwargs = {"classes": np.array([0, 1])} if first_batch else {}
        reranker.partial_fit(batch_x, batch_y, sample_weight=sample_weight, **kwargs)
        first_batch = False
        del batch_x, batch_y, sample_weight


def supervised_ranking(pos, row, limit=TOP_K):
    candidates, feats = candidate_features(pos, row)
    pred = reranker.predict_proba(feats)[:, 1]
    return candidates[np.argsort(pred)[::-1][:limit]].tolist()


def supervised_builder(pos, row):
    return supervised_ranking(pos, row, TOP_K)


evaluate("logistic supervised reranker", supervised_builder, test_positions, "test")

# The hard location quota was useful for the hand-tuned fusion, but the model
# already sees location features.  Test explicitly instead of assuming that the
# same post-processing still helps.
for quota in (20, 30, 40):
    evaluate(
        f"logistic reranker + exact-location quota {quota}",
        lambda p, r, q=quota: quota_ranking(
            supervised_ranking(p, r, RETRIEVAL_K),
            item_locs,
            int(r["search_location_id"]),
            q,
        ),
        test_positions,
        "test",
    )

# A realistic history diagnostic: build the history channel only from tune
# labels and evaluate transfer to different held-out contexts.  This avoids
# giving a test row its own clicked item while mimicking the information that
# is available for the real benchmark.
history_counts = defaultdict(lambda: defaultdict(int))
for pos in tune_positions:
    query = val_main.iloc[pos]
    for idx in validation.iloc[pos]["relevant_idx"]:
        history_counts[query][int(idx)] += 1
history_rankings = {
    query: [idx for idx, _ in sorted(counts.items(), key=lambda pair: pair[1], reverse=True)]
    for query, counts in history_counts.items()
}


def history_fusion_builder(history_weight, quota=None):
    def builder(pos, row):
        learned = supervised_ranking(pos, row, RETRIEVAL_K)
        historical = history_rankings.get(val_main.iloc[pos], [])
        scores = rrf_score([learned, historical], [1.0, history_weight])
        deep = rank_from_scores(scores, RETRIEVAL_K)
        if quota is not None:
            deep = quota_ranking(deep, item_locs, int(row["search_location_id"]), quota)
        return deep[:TOP_K]

    return builder


for weight in (0.5, 1.0, 2.0):
    evaluate(
        f"logistic + leakage-safe history RRF weight {weight}",
        history_fusion_builder(weight),
        test_positions,
        "test",
    )

# Diagnostics by query-history and exact-location availability.
item_location_set = set(map(int, item_locs))
best_builder = make_builder(best_config, best_kwargs)
segments = {
    "has_exact_query_history": [p for p in test_positions if val_main.iloc[p] in exact_micro_probs],
    "no_exact_query_history": [p for p in test_positions if val_main.iloc[p] not in exact_micro_probs],
    "exact_location_exists": [p for p in test_positions if int(validation.iloc[p]["search_location_id"]) in item_location_set],
    "exact_location_absent": [p for p in test_positions if int(validation.iloc[p]["search_location_id"]) not in item_location_set],
}
for segment, positions in segments.items():
    if positions:
        evaluate(f"best segment: {segment} (n={len(positions)})", best_builder, positions, "segment")

output = {
    "settings": {
        "seed": SEED,
        "groups": len(validation),
        "tune_groups": len(tune_positions),
        "test_groups": len(test_positions),
        "retrieval_k": RETRIEVAL_K,
        "top_k": TOP_K,
        "best_text_config": best_config,
        "best_structured_config": best_structured_name,
        "runtime_seconds": time.time() - started,
    },
    "coverage": {
        "validation_exact_query_profile": float(val_main.isin(exact_micro_probs).mean()),
        "validation_exact_location_in_items": float(validation["search_location_id"].isin(item_location_set).mean()),
    },
    "reranker": {
        "feature_order": [
            *[part for name in all_names for part in (f"{name}_reciprocal_rank", f"{name}_cosine")],
            "exact_location",
            "location_transition_probability",
            "exact_query_microcat_probability",
            "microcat_centroid_similarity",
            "popularity",
            "log_reviews",
            "rating",
        ],
        "coef": reranker.coef_[0].astype(float).tolist(),
        "intercept": reranker.intercept_.astype(float).tolist(),
    },
    "results": records,
}
OUT.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {OUT}; runtime={time.time() - started:.1f}s", flush=True)


In [ ]:
"""Generate the v3 Avito candidate-retrieval submission.

The script reuses the leakage-safe linear reranker fitted in
``hybrid_experiments.py``.  It retrieves a broad union of five lexical
channels, scores candidates with learned structural features, softly fuses
historical clicks, and applies a small exact-location quota.

Peak memory is kept below the earlier implementation by materialising one
sparse item field at a time.
"""

from __future__ import annotations

from collections import defaultdict
import gc
import json
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


DATA = train_path.parent
MODEL = working / "hybrid_lora_results.json"
DENSE = output_npz
OUTPUT = working / "answer.csv"
RETRIEVAL_K = 180
TOP_K = 50


def normalize(value) -> str:
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value)).lower().replace("ё", "е")
    value = re.sub(r"[^0-9a-zа-я]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def clean(series: pd.Series) -> pd.Series:
    return series.fillna("").map(normalize)


def sparse_topk(qmat, imat, k=RETRIEVAL_K, batch=64):
    """Return item positions and cosine scores for each sparse query row."""
    all_idx, all_val = [], []
    transposed = imat.T.tocsc()
    for start in range(0, qmat.shape[0], batch):
        scores = (qmat[start : start + batch] @ transposed).tocsr()
        for row in range(scores.shape[0]):
            left, right = scores.indptr[row], scores.indptr[row + 1]
            idx = scores.indices[left:right]
            val = scores.data[left:right]
            if len(val) > k:
                keep = np.argpartition(val, -k)[-k:]
                keep = keep[np.argsort(val[keep])[::-1]]
                idx, val = idx[keep], val[keep]
            elif len(val):
                order = np.argsort(val)[::-1]
                idx, val = idx[order], val[order]
            all_idx.append(idx.astype(np.int32, copy=False))
            all_val.append(val.astype(np.float32, copy=False))
    return all_idx, all_val


def rrf(rankings, weights, limit=RETRIEVAL_K, rrf_k=60):
    scores = defaultdict(float)
    for ranking, weight in zip(rankings, weights):
        for rank, idx in enumerate(ranking):
            scores[int(idx)] += weight / (rrf_k + rank + 1)
    return [idx for idx, _ in sorted(scores.items(), key=lambda pair: pair[1], reverse=True)[:limit]]


def location_quota(ranking, item_locations, query_location, quota=20):
    """Reserve up to ``quota`` places for exact-location candidates."""
    local = [idx for idx in ranking if item_locations[idx] == query_location]
    chosen = local[:quota]
    used = set(chosen)
    for idx in ranking:
        if idx not in used:
            chosen.append(idx)
            used.add(idx)
        if len(chosen) == TOP_K:
            break
    return chosen


print("Loading data and learned coefficients", flush=True)
model = json.loads(MODEL.read_text(encoding="utf-8"))["reranker"]
coef = np.asarray(model["coef"], dtype=np.float32)
intercept = float(model["intercept"][0])

train_cols = [
    "search_query", "search_location_id", "item_id", "item_location_id",
    "item_microcat_id",
]
train = pd.read_parquet(DATA / "train.parquet", columns=train_cols)
queries = pd.read_parquet(DATA / "benchmark_queries.parquet")
items = pd.read_parquet(DATA / "benchmark_items.parquet")

item_ids = items["item_id"].astype(str).to_numpy()
item_id_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids)}
benchmark_item_ids = set(item_ids)
item_locations = items["item_location_id"].to_numpy()
item_microcats = items["item_microcat_id"].to_numpy()

query_main = clean(queries["search_query"])
query_params = clean(queries["search_infm_params_text"])
query_mixed = query_main + " " + query_main + " " + query_params

print("Building leakage-safe behaviour profiles", flush=True)
profile = train.loc[
    ~train["item_id"].isin(benchmark_item_ids),
    ["search_query", "search_location_id", "item_location_id", "item_microcat_id"],
].copy()
profile["normalized_query"] = clean(profile["search_query"])

loc_counts = (
    profile.groupby(["search_location_id", "item_location_id"])
    .size().rename("n").reset_index()
)
loc_counts["total"] = loc_counts.groupby("search_location_id")["n"].transform("sum")
loc_counts["prob"] = loc_counts["n"] / loc_counts["total"]
loc_counts = loc_counts.sort_values(["search_location_id", "n"], ascending=[True, False])
transition_probs = {
    int(source): {int(target): float(prob) for target, prob in zip(group.item_location_id, group.prob)}
    for source, group in loc_counts.groupby("search_location_id").head(5).groupby("search_location_id")
}

micro_counts = (
    profile.groupby(["normalized_query", "item_microcat_id"])
    .size().rename("n").reset_index()
)
micro_counts["total"] = micro_counts.groupby("normalized_query")["n"].transform("sum")
micro_counts["prob"] = micro_counts["n"] / micro_counts["total"]
micro_counts = micro_counts.sort_values(["normalized_query", "n"], ascending=[True, False])
exact_micro_probs = {
    query: {int(cat): float(prob) for cat, prob in zip(group.item_microcat_id, group.prob)}
    for query, group in micro_counts.groupby("normalized_query").head(5).groupby("normalized_query")
}

# Query-to-microcategory centroids cover unseen queries.
query_cat_counts = (
    profile.groupby(["item_microcat_id", "normalized_query"])
    .size().rename("n").reset_index()
    .sort_values(["item_microcat_id", "n"], ascending=[True, False])
)
micro_docs = (
    query_cat_counts.groupby("item_microcat_id").head(250)
    .groupby("item_microcat_id")["normalized_query"].agg(" ".join)
)
micro_vec = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=2,
    max_features=80_000, sublinear_tf=True, dtype=np.float32,
)
micro_mat = micro_vec.fit_transform(micro_docs)
micro_similarity = (micro_vec.transform(query_main) @ micro_mat.T).tocsr()
microcat_values = micro_docs.index.to_numpy()
centroid_by_query = []
for row in range(micro_similarity.shape[0]):
    left, right = micro_similarity.indptr[row], micro_similarity.indptr[row + 1]
    values = micro_similarity.data[left:right]
    columns = micro_similarity.indices[left:right]
    if len(values) > 5:
        keep = np.argpartition(values, -5)[-5:]
        values, columns = values[keep], columns[keep]
    centroid_by_query.append(
        {int(microcat_values[col]): float(value) for col, value in zip(columns, values)}
    )
del micro_vec, micro_mat, micro_similarity, query_cat_counts, micro_docs

# Historical benchmark items are a separate, softly fused channel.  They are
# supplied training evidence, not reconstructed test labels.
history = train.loc[train["item_id"].isin(benchmark_item_ids), ["search_query", "item_id"]].copy()
history["normalized_query"] = clean(history["search_query"])
history_counts = (
    history.groupby(["normalized_query", "item_id"]).size().rename("n").reset_index()
    .sort_values(["normalized_query", "n"], ascending=[True, False])
)
history_lookup = {
    query: [item_id_to_idx[item_id] for item_id in group["item_id"]]
    for query, group in history_counts.groupby("normalized_query", sort=False)
}
del history, history_counts, train
gc.collect()

print("Preparing item text", flush=True)
item_title = clean(items["item_title_raw"])
item_params = clean(items["item_infm_params_text"])
item_description = clean(items["item_description_raw"]).str.slice(0, 1000)
item_mixed = item_title + " " + item_title + " " + item_params + " " + item_description
items.drop(columns=["item_title_raw", "item_infm_params_text", "item_description_raw"], inplace=True)

channels = {}
channel_scores = {}
print("Fitting word index", flush=True)
word_vec = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.995,
    max_features=200_000, sublinear_tf=True, dtype=np.float32,
)
word_vec.fit(item_mixed)
for name, query_text, item_text in [
    ("mixed", query_mixed, item_mixed),
    ("title", query_main, item_title),
    ("params", query_main + " " + query_params + " " + query_params, item_params),
    ("description", query_main, item_description),
]:
    print(f"Retrieving {name}", flush=True)
    query_matrix = word_vec.transform(query_text)
    item_matrix = word_vec.transform(item_text)
    channels[name], channel_scores[name] = sparse_topk(query_matrix, item_matrix)
    del query_matrix, item_matrix
    gc.collect()
    if name == "mixed":
        del item_mixed
del word_vec
gc.collect()

print("Retrieving char n-grams", flush=True)
char_vec = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.995,
    max_features=180_000, sublinear_tf=True, dtype=np.float32,
)
char_items = char_vec.fit_transform(item_title + " " + item_params.str.slice(0, 300))
char_queries = char_vec.transform(query_main)
channels["char"], channel_scores["char"] = sparse_topk(char_queries, char_items)
del char_vec, char_items, char_queries, item_title, item_params, item_description
gc.collect()

print("Loading ML-Embed benchmark candidates", flush=True)
dense = np.load(DENSE, allow_pickle=True)  # trusted artifact produced by our Kaggle notebook
assert np.array_equal(dense["item_ids"].astype(str), item_ids)
assert np.array_equal(
    dense["benchmark_query_ids"].astype(str),
    queries["query_id"].astype(str).to_numpy(),
)
channels["dense"] = [row.astype(np.int32, copy=False) for row in dense["benchmark_indices"]]
channel_scores["dense"] = [row.astype(np.float32) for row in dense["benchmark_scores"]]

reviews = items["item_rating_reviews_count"].fillna(0).to_numpy(dtype=float)
ratings = items["item_rating"].fillna(0).to_numpy(dtype=float)
popularity = np.log1p(reviews) * np.clip(ratings / 5, 0, 1)
popularity /= max(popularity.max(), 1)
review_norm = np.log1p(reviews)
review_norm /= max(review_norm.max(), 1)
fallback = np.argsort(popularity)[::-1].tolist()

all_names = ["mixed", "title", "params", "description", "char", "dense"]
expected_features = [
    *[part for name in all_names for part in (f"{name}_reciprocal_rank", f"{name}_cosine")],
    "exact_location", "location_transition_probability",
    "exact_query_microcat_probability", "microcat_centroid_similarity",
    "popularity", "log_reviews", "rating",
]
assert model["feature_order"] == expected_features
assert len(coef) == len(expected_features)

print("Scoring and fusing candidates", flush=True)
final_rankings = []
for pos, row in queries.iterrows():
    candidate_set = set()
    rank_maps, score_maps = {}, {}
    for name in all_names:
        ranking = channels[name][pos]
        values = channel_scores[name][pos]
        candidate_set.update(map(int, ranking))
        rank_maps[name] = {int(idx): rank for rank, idx in enumerate(ranking)}
        score_maps[name] = {int(idx): float(value) for idx, value in zip(ranking, values)}

    source_location = int(row["search_location_id"])
    transitions = transition_probs.get(source_location, {})
    normalized_query = query_main.iloc[pos]
    exact_micro = exact_micro_probs.get(normalized_query, {})
    centroid_micro = centroid_by_query[pos]
    candidates = np.fromiter(candidate_set, dtype=np.int32)
    features = []
    for idx in candidates:
        values = []
        for name in all_names:
            rank = rank_maps[name].get(int(idx), RETRIEVAL_K + 50)
            values.extend([1.0 / (rank + 1), score_maps[name].get(int(idx), 0.0)])
        values.extend([
            float(item_locations[idx] == source_location),
            transitions.get(int(item_locations[idx]), 0.0),
            exact_micro.get(int(item_microcats[idx]), 0.0),
            centroid_micro.get(int(item_microcats[idx]), 0.0),
            popularity[idx], review_norm[idx], ratings[idx] / 5,
        ])
        features.append(values)
    features = np.asarray(features, dtype=np.float32)
    learned_scores = features @ coef + intercept
    learned = candidates[np.argsort(learned_scores)[::-1][:RETRIEVAL_K]].tolist()

    # Weight 1.0 was the best conservative setting on leakage-safe holdout.
    historical = history_lookup.get(normalized_query, [])
    fused = rrf([learned, historical], [1.0, 1.0], RETRIEVAL_K)
    chosen = location_quota(fused, item_locations, source_location, quota=20)

    seen = set(chosen)
    if len(chosen) < TOP_K:
        for idx in fallback:
            if idx not in seen:
                chosen.append(idx)
                seen.add(idx)
            if len(chosen) == TOP_K:
                break
    final_rankings.append(chosen[:TOP_K])

answer = pd.DataFrame({
    "query_id": queries["query_id"].astype(str),
    "answer": [" ".join(item_ids[ranking]) for ranking in final_rankings],
})
answer.to_csv(OUTPUT, index=False, encoding="utf-8")

# Strict submission checks.  IDs must remain strings to preserve leading zeros.
saved = pd.read_csv(OUTPUT, dtype=str, keep_default_na=False)
assert saved.columns.tolist() == ["query_id", "answer"]
assert len(saved) == len(queries)
assert saved["query_id"].tolist() == queries["query_id"].astype(str).tolist()
assert saved["query_id"].is_unique
for query_id, value in saved.itertuples(index=False):
    ids = value.split(" ")
    assert len(ids) == TOP_K
    assert len(ids) == len(set(ids))
    assert set(ids) <= benchmark_item_ids
    assert all(re.fullmatch(r"[0-9a-f]{16}", item_id) for item_id in ids), query_id

print(f"{OUTPUT} is valid: {len(saved)} rows, exactly {TOP_K} IDs per row", flush=True)


## 8. Финальная проверка

`answer.csv` содержит по одной строке на каждый benchmark query, две колонки,
ровно 50 уникальных `item_id` в нужном регистре. Генератор завершается
assert-проверками всех этих условий.


In [ ]:
# Optional cleanup of large temporary shards and downloaded data.
for r in range(2):
    for prefix in ["item_emb_rank", "query_emb_rank"]:
        p = working / f"{prefix}{r}.npy"
        if p.exists():
            p.unlink()

for p in [
    working / "lora_train_pairs.parquet",
    working / "lora_item_texts.parquet",
    working / "lora_query_texts.parquet",
    working / "train_lora_ddp.py",
    working / "encode_lora_ddp.py",
]:
    if p.exists():
        p.unlink()

for p in [
    working / "avito_data",
    working / "dataset_outer.zip",
]:
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()

print("Temporary files removed.")